In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.listdir('/content/drive/MyDrive/sales_analysis_project/data')


Mounted at /content/drive


['sales_analysis_dataset.csv', 'cleaned_sales.csv']

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/sales_analysis_project/data/sales_analysis_dataset.csv"
df = pd.read_csv(file_path)

df.head()

,Order ID,Order Date,Region,Category,Product,Sales,Profit,Quantity,Customer
0,ORD1000,2023-01-01,South,Technology,Chair,761.84,234.56,5,Customer_C
1,ORD1001,2023-01-02,Central,Technology,Storage,1879.63,43.19,4,Customer_A
2,ORD1002,2023-01-03,East,Furniture,Binder,174.45,-53.61,8,Customer_C
3,ORD1003,2023-01-04,South,Furniture,Laptop,894.10,205.75,6,Customer_B
4,ORD1004,2023-01-05,South,Office Supplies,Phone,1938.40,39.09,6,Customer_C


In [ ]:
#understand data

df.info()
print(df.describe())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Order ID    200 non-null    object 
 1   Order Date  200 non-null    object 
 2   Region      200 non-null    object 
 3   Category    200 non-null    object 
 4   Product     200 non-null    object 
 5   Sales       200 non-null    float64
 6   Profit      200 non-null    float64
 7   Quantity    200 non-null    int64  
 8   Customer    200 non-null    object 
dtypes: float64(2), int64(1), object(6)
memory usage: 14.2+ KB
             Sales      Profit    Quantity
count   200.000000  200.000000  200.000000
mean   1072.955300  157.412000    4.985000
std     545.443523  190.125556    2.590042
min     125.980000 -198.180000    1.000000
25%     602.435000   31.197500    3.000000
50%    1085.080000  174.630000    5.000000
75%    1528.197500  307.367500    7.000000
max    1992.270000  498.380000    9.00

In [ ]:
#cleaning data

df.duplicated().sum()
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Order Date']

#create new columns
df['Year'] = df['Order Date'].dt.year
df['Year']
df['Month'] = df['Order Date'].dt.month
df['Month']
df['Month Name'] = df['Order Date'].dt.month_name()
df['Month Name']
df.head(5)

#create profit margin
df['Profit Margin'] = df['Profit'] / df['Sales']
df['Profit Margin']
print(df.head())

#data validation
print(df[df['Sales'] <= 0])
print(df[df['Profit'].isnull()])



  Order ID Order Date   Region         Category  Product    Sales  Profit  \
0  ORD1000 2023-01-01    South       Technology    Chair   761.84  234.56   
1  ORD1001 2023-01-02  Central       Technology  Storage  1879.63   43.19   
2  ORD1002 2023-01-03     East        Furniture   Binder   174.45  -53.61   
3  ORD1003 2023-01-04    South        Furniture   Laptop   894.10  205.75   
4  ORD1004 2023-01-05    South  Office Supplies    Phone  1938.40   39.09   

   Quantity    Customer  Year  Month Month Name  Profit Margin  
0         5  Customer_C  2023      1    January       0.307886  
1         4  Customer_A  2023      1    January       0.022978  
2         8  Customer_C  2023      1    January      -0.307309  
3         6  Customer_B  2023      1    January       0.230120  
4         6  Customer_C  2023      1    January       0.020166  
Empty DataFrame
Columns: [Order ID, Order Date, Region, Category, Product, Sales, Profit, Quantity, Customer, Year, Month, Month Name, Profit Margi

In [ ]:
#exploratory data analysis
print("Total Sales:", df['Sales'].sum())
print("Total Profit:", df['Profit'].sum())

#sales by category
df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

#sales by region
df.groupby('Region')['Sales'].sum().sort_values(ascending=False)

#monthly sales trend
df.groupby('Month Name')['Sales'].sum()

#top 5 customers
df.groupby('Customer')['Sales'].sum().sort_values(ascending=False).head()

#top products
df.groupby('Product')['Sales'].sum().sort_values(ascending=False)

Total Sales: 214591.06
Total Profit: 31482.4


,Sales
Product,
Table,47161.39
Binder,37264.06
Storage,36648.19
Phone,35865.23
Chair,34658.06
Laptop,22994.13


In [ ]:
#save cleaned data
df.to_csv("/content/drive/MyDrive/sales_analysis_project/data/cleaned_sales.csv", index=False)

In [ ]:
#load cleaned data
import pandas as pd

file_path = "/content/drive/MyDrive/sales_analysis_project/data/cleaned_sales.csv"
df = pd.read_csv(file_path)

df.head()

,Order ID,Order Date,Region,Category,Product,Sales,Profit,Quantity,Customer,Year,Month,Month Name,Profit Margin
0,ORD1000,2023-01-01,South,Technology,Chair,761.84,234.56,5,Customer_C,2023,1,January,0.307886
1,ORD1001,2023-01-02,Central,Technology,Storage,1879.63,43.19,4,Customer_A,2023,1,January,0.022978
2,ORD1002,2023-01-03,East,Furniture,Binder,174.45,-53.61,8,Customer_C,2023,1,January,-0.307309
3,ORD1003,2023-01-04,South,Furniture,Laptop,894.10,205.75,6,Customer_B,2023,1,January,0.230120
4,ORD1004,2023-01-05,South,Office Supplies,Phone,1938.40,39.09,6,Customer_C,2023,1,January,0.020166


In [ ]:
#load cleaned data into sql
import sqlite3

conn = sqlite3.connect('sales.db')
df.to_sql('sales', conn, if_exists='replace', index=False)

#check data
query = "SELECT * FROM sales LIMIT 5"
pd.read_sql(query, conn)

#total sales
query = "SELECT SUM(Sales) AS Total_Sales FROM sales"
pd.read_sql(query, conn)

#sales by category
query = """
SELECT Category, SUM(Sales) AS Total_Sales
FROM sales
GROUP BY Category
ORDER BY Total_Sales DESC
"""
pd.read_sql(query, conn)

#profit by region
query = """
SELECT Region, SUM(Profit) AS Total_Profit
FROM sales
GROUP BY Region
ORDER BY Total_Profit DESC
"""
print(pd.read_sql(query,conn))

#Top 5 products
query = """
SELECT Product, SUM(Sales) AS Total_Sales
FROM sales
GROUP BY Product
ORDER BY Total_Sales DESC
LIMIT 5
"""
pd.read_sql(query,conn)

#monthly sales trend
query = """
SELECT Month, SUM(Sales) AS Monthly_Sales
FROM sales
GROUP BY Month
ORDER BY Month
"""
pd.read_sql(query,conn)

#high profit orders
query = """
SELECT *
FROM sales
WHERE Profit > 300
"""
pd.read_sql(query,conn)

    Region  Total_Profit
0    South       9231.26
1     West       7914.42
2     East       7896.17
3  Central       6440.55


,Order ID,Order Date,Region,Category,Product,Sales,Profit,Quantity,Customer,Year,Month,Month Name,Profit Margin
0,ORD1013,2023-01-14,South,Technology,Table,1747.52,412.02,4,Customer_A,2023,1,January,0.235774
1,ORD1015,2023-01-16,East,Technology,Table,553.33,481.18,8,Customer_B,2023,1,January,0.869608
2,ORD1018,2023-01-19,Central,Office Supplies,Table,552.65,375.25,1,Customer_B,2023,1,January,0.679001
3,ORD1025,2023-01-26,West,Office Supplies,Chair,201.62,425.95,7,Customer_C,2023,1,January,2.112638
4,ORD1036,2023-02-06,Central,Office Supplies,Binder,1152.04,407.81,3,Customer_C,2023,2,February,0.353989
5,ORD1039,2023-02-09,South,Furniture,Chair,1584.46,490.38,9,Customer_A,2023,2,February,0.309493
6,ORD1041,2023-02-11,South,Technology,Binder,1868.77,446.83,9,Customer_C,2023,2,February,0.239104
7,ORD1043,2023-02-13,Central,Technology,Chair,1992.27,331.97,9,Customer_C,2023,2,February,0.166629
8,ORD1045,2023-02-15,Central,Office Supplies,Storage,1500.37,304.36,9,Customer_B,2023,2,February,0.202857
9,ORD1050,2023-02-20,South,Technology,Table,1690.23,391.11,2,Customer_B,2023,2,February,0.231395
